In [1]:
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score
from sentence_transformers import SentenceTransformer


c:\Users\titou\Documents\dev\Ynov\DataScience\Mini-memoire-Base-vectorielles\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ==========================================
# 1. Chargement et Préparation des données
# ==========================================
print("1. Chargement des données...")
# Remplacez 'votre_fichier.csv' par le bon chemin
df = pd.read_csv('C:/Users/titou/Documents/dev/Ynov/DataScience/Mini-memoire-Base-vectorielles/Data/train_tm/train.csv')

# pour ne prendre que 3000 articles aléatoirement :
# df = df.sample(3000, random_state=42)

# On fusionne le TITRE et l'ABSTRACT pour avoir un texte riche en contexte
df['TEXTE_COMPLET'] = df['TITLE'] + " " + df['ABSTRACT']

# On isole nos variables d'entrée (X) et nos variables cibles (y)
X_textes = df['TEXTE_COMPLET'].tolist()
labels_cols = ['Computer Science', 'Physics', 'Mathematics', 'Statistics', 'Quantitative Biology', 'Quantitative Finance']
y_labels = df[labels_cols].values

# Séparation en jeu d'entraînement (80%) et jeu de test (20%)
X_train_text, X_test_text, y_train, y_test = train_test_split(X_textes, y_labels, test_size=0.2, random_state=42)

# On prépare le modèle d'apprentissage :
# Comme un article peut avoir plusieurs tags (multi-label), on utilise OneVsRestClassifier
modele_ml = OneVsRestClassifier(LogisticRegression(max_iter=1000))

# ==========================================
# EXPÉRIENCE A : L'approche classique TF-IDF
# ==========================================
print("\n--- Début de l'Expérience TF-IDF ---")
start_time_tfidf = time.time()

# 1. Vectorisation TF-IDF
# On limite à 10000 mots maximum pour ne pas exploser la mémoire
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

# 2. Entraînement et Prédiction
modele_ml.fit(X_train_tfidf, y_train)
predictions_tfidf = modele_ml.predict(X_test_tfidf)

temps_tfidf = time.time() - start_time_tfidf
# Le Score F1 "Micro" calcule la moyenne globale des bonnes prédictions multi-labels
score_tfidf = f1_score(y_test, predictions_tfidf, average='micro')

print(f"TF-IDF terminé en {temps_tfidf:.2f} secondes.")
print(f"Score F1 (TF-IDF) : {score_tfidf:.4f}")

# ==========================================
# EXPÉRIENCE B : L'approche moderne BERT
# ==========================================
print("\n--- Début de l'Expérience BERT ---")
start_time_bert = time.time()

# 1. Vectorisation avec Sentence-Transformers
# Le modèle va comprendre le contexte global de chaque texte de recherche
encodeur_bert = SentenceTransformer('all-MiniLM-L6-v2')

print("Encodage des textes en cours (cela peut prendre un peu de temps)...")
X_train_bert = encodeur_bert.encode(X_train_text, show_progress_bar=True)
X_test_bert = encodeur_bert.encode(X_test_text, show_progress_bar=True)

# 2. Entraînement et Prédiction (sur les vecteurs denses produits par BERT)
modele_ml.fit(X_train_bert, y_train)
predictions_bert = modele_ml.predict(X_test_bert)

temps_bert = time.time() - start_time_bert
score_bert = f1_score(y_test, predictions_bert, average='micro')

print(f"BERT terminé en {temps_bert:.2f} secondes.")
print(f"Score F1 (BERT) : {score_bert:.4f}")

# ==========================================
# BILAN ET COMPARAISON POUR LE MEMOIRE
# ==========================================
print("\n==========================================")
print("BILAN DE LA COMPARAISON")
print("==========================================")
print(f"Modèle  | Score F1 | Temps d'exécution")
print(f"TF-IDF  |  {score_tfidf:.4f}  | {temps_tfidf:.2f} sec")
print(f"BERT    |  {score_bert:.4f}  | {temps_bert:.2f} sec")

difference_perf = (score_bert - score_tfidf) * 100
print(f"\nConclusion : BERT améliore les performances de {difference_perf:.2f} points de pourcentage.")
print("Cependant, il nécessite beaucoup plus de temps de calcul pour créer les embeddings.")

1. Chargement des données...

--- Début de l'Expérience TF-IDF ---
TF-IDF terminé en 2.69 secondes.
Score F1 (TF-IDF) : 0.7982

--- Début de l'Expérience BERT ---


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3952.84it/s]


Encodage des textes en cours (cela peut prendre un peu de temps)...


Batches: 100%|██████████| 132/132 [01:25<00:00,  1.55it/s]


BERT terminé en 427.51 secondes.
Score F1 (BERT) : 0.8136

BILAN DE LA COMPARAISON
Modèle  | Score F1 | Temps d'exécution
TF-IDF  |  0.7982  | 2.69 sec
BERT    |  0.8136  | 427.51 sec

Conclusion : BERT améliore les performances de 1.54 points de pourcentage.
Cependant, il nécessite beaucoup plus de temps de calcul pour créer les embeddings.


In [3]:

# ==========================================
# 1. Chargement et Préparation des données
# ==========================================
print("1. Chargement des données...")
# Remplacez 'votre_fichier.csv' par le bon chemin
df = pd.read_csv('C:/Users/titou/Documents/dev/Ynov/DataScience/Mini-memoire-Base-vectorielles/Data/train_tm/train.csv')

# ASTUCE POUR LE MEMOIRE : L'encodage de 20 000 textes avec BERT sur un processeur classique (CPU) 
# peut prendre 10 à 15 minutes. Pour faire vos tests au début, décommentez la ligne suivante 
# pour ne prendre que 3000 articles aléatoirement :
# df = df.sample(3000, random_state=42)

# On fusionne le TITRE et l'ABSTRACT pour avoir un texte riche en contexte
df['TEXTE_COMPLET'] = df['TITLE'] + " " + df['ABSTRACT']

# On isole nos variables d'entrée (X) et nos variables cibles (y)
X_textes = df['TEXTE_COMPLET'].tolist()
labels_cols = ['Computer Science', 'Physics', 'Mathematics', 'Statistics', 'Quantitative Biology', 'Quantitative Finance']
y_labels = df[labels_cols].values

# Séparation en jeu d'entraînement (80%) et jeu de test (20%)
X_train_text, X_test_text, y_train, y_test = train_test_split(X_textes, y_labels, test_size=0.2, random_state=42)

# On prépare le modèle d'apprentissage :
# Comme un article peut avoir plusieurs tags (multi-label), on utilise OneVsRestClassifier
modele_ml = OneVsRestClassifier(LogisticRegression(max_iter=1000))

# ==========================================
# EXPÉRIENCE A : L'approche classique TF-IDF
# ==========================================
print("\n--- Début de l'Expérience TF-IDF ---")
start_time_tfidf = time.time()

# 1. Vectorisation TF-IDF
# On limite à 10000 mots maximum pour ne pas exploser la mémoire
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

# 2. Entraînement et Prédiction
modele_ml.fit(X_train_tfidf, y_train)
predictions_tfidf = modele_ml.predict(X_test_tfidf)

temps_tfidf = time.time() - start_time_tfidf
# Le Score F1 "Micro" calcule la moyenne globale des bonnes prédictions multi-labels
score_tfidf = f1_score(y_test, predictions_tfidf, average='micro')

print(f"TF-IDF terminé en {temps_tfidf:.2f} secondes.")
print(f"Score F1 (TF-IDF) : {score_tfidf:.4f}")

# ==========================================
# EXPÉRIENCE B : L'approche moderne BERT
# ==========================================
print("\n--- Début de l'Expérience BERT ---")
start_time_bert = time.time()

# 1. Vectorisation avec Sentence-Transformers
# Le modèle va comprendre le contexte global de chaque texte de recherche
encodeur_bert = SentenceTransformer('all-MiniLM-L6-v2')

print("Encodage des textes en cours (cela peut prendre un peu de temps)...")
X_train_bert = encodeur_bert.encode(X_train_text, show_progress_bar=True)
X_test_bert = encodeur_bert.encode(X_test_text, show_progress_bar=True)

# 2. Entraînement et Prédiction (sur les vecteurs denses produits par BERT)
modele_ml.fit(X_train_bert, y_train)
predictions_bert = modele_ml.predict(X_test_bert)

temps_bert = time.time() - start_time_bert
score_bert = f1_score(y_test, predictions_bert, average='micro')

print(f"BERT terminé en {temps_bert:.2f} secondes.")
print(f"Score F1 (BERT) : {score_bert:.4f}")

# ==========================================
# BILAN ET COMPARAISON POUR LE MEMOIRE
# ==========================================
print("\n==========================================")
print("BILAN DE LA COMPARAISON")
print("==========================================")
print(f"Modèle  | Score F1 | Temps d'exécution")
print(f"TF-IDF  |  {score_tfidf:.4f}  | {temps_tfidf:.2f} sec")
print(f"BERT    |  {score_bert:.4f}  | {temps_bert:.2f} sec")

difference_perf = (score_bert - score_tfidf) * 100
print(f"\nConclusion : BERT améliore les performances de {difference_perf:.2f} points de pourcentage.")
print("Cependant, il nécessite beaucoup plus de temps de calcul pour créer les embeddings.")

1. Chargement des données...

--- Début de l'Expérience TF-IDF ---
TF-IDF terminé en 2.70 secondes.
Score F1 (TF-IDF) : 0.7982

--- Début de l'Expérience BERT ---


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5006.64it/s]


Encodage des textes en cours (cela peut prendre un peu de temps)...


Batches: 100%|██████████| 132/132 [01:24<00:00,  1.56it/s]


BERT terminé en 427.20 secondes.
Score F1 (BERT) : 0.8136

BILAN DE LA COMPARAISON
Modèle  | Score F1 | Temps d'exécution
TF-IDF  |  0.7982  | 2.70 sec
BERT    |  0.8136  | 427.20 sec

Conclusion : BERT améliore les performances de 1.54 points de pourcentage.
Cependant, il nécessite beaucoup plus de temps de calcul pour créer les embeddings.


In [10]:
from sklearn.decomposition import PCA

# ==========================================
# EXPÉRIENCE C : BERT + PCA
# ==========================================
print("\n--- Début de l'Expérience BERT avec PCA ---")
start_time_pca = time.time()

# 1. Configuration de la PCA
# Au lieu de donner un nombre de dimensions fixe, on donne un float (ex: 0.95).
# Cela indique à la PCA : "Trouve le nombre minimum de dimensions nécessaires 
# pour conserver 95% de l'information (variance) des embeddings originaux".
pca = PCA(n_components=0.95, random_state=42)

print("Application de la PCA en cours...")
# On "fit" (calcule les axes) UNIQUEMENT sur le jeu d'entraînement pour éviter la fuite de données (data leakage)
X_train_bert_pca = pca.fit_transform(X_train_bert)
# On applique la même transformation sur le jeu de test
X_test_bert_pca = pca.transform(X_test_bert)

# Affichage de la réduction pour votre analyse
dim_originale = X_train_bert.shape[1]
dim_reduite = X_train_bert_pca.shape[1]
print(f"Dimension réduite de {dim_originale} à {dim_reduite} composantes (pour 95% d'information conservée).")

# 2. Entraînement et Prédiction sur les données réduites
modele_ml.fit(X_train_bert_pca, y_train)
predictions_bert_pca = modele_ml.predict(X_test_bert_pca)

temps_pca = time.time() - start_time_pca
score_pca = f1_score(y_test, predictions_bert_pca, average='micro')

print(f"BERT + PCA terminé en {temps_pca:.2f} secondes.")
print(f"Score F1 (BERT + PCA) : {score_pca:.4f}")

# ==========================================
# NOUVEAU BILAN COMPARATIF
# ==========================================
print("\n==========================================")
print("BILAN GLOBAL DE LA COMPARAISON")
print("==========================================")
print(f"Modèle       | Score F1 | Temps d'exécution")
print(f"TF-IDF       |  {score_tfidf:.4f}  | {temps_tfidf:.2f} sec")
print(f"BERT pur     |  {score_bert:.4f}  | {temps_bert:.2f} sec")
print(f"BERT + PCA   |  {score_pca:.4f}  | {temps_pca:.2f} sec")


--- Début de l'Expérience BERT avec PCA ---
Application de la PCA en cours...
Dimension réduite de 384 à 232 composantes (pour 95% d'information conservée).
BERT + PCA terminé en 0.55 secondes.
Score F1 (BERT + PCA) : 0.8126

BILAN GLOBAL DE LA COMPARAISON
Modèle       | Score F1 | Temps d'exécution
TF-IDF       |  0.7982  | 3.35 sec
BERT pur     |  0.8136  | 437.75 sec
BERT + PCA   |  0.8126  | 0.55 sec


In [ ]:
import time
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.decomposition import PCA
import umap.umap_ as umap
from sentence_transformers import SentenceTransformer

print("1. Chargement des données (Espace vs Médecine)...")
categories = ['sci.med', 'sci.space']
dataset = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

# On prend 1000 textes pour l'exemple (à adapter selon la puissance de votre machine)
textes = dataset.data[:1000]
labels = dataset.target[:1000]

X_train_text, X_test_text, y_train, y_test = train_test_split(textes, labels, test_size=0.2, random_state=42)

# =========================================================
# EXPERIMENTATION 1 : NOMIC BASELINE (768 dimensions)
# =========================================================
print("\n2. Chargement du modèle Nomic...")
# trust_remote_code=True est obligatoire pour Nomic v1.5
model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)

# Ajout du préfixe requis par Nomic pour les tâches de classification
print("Encodage des textes (Baseline à 768 dimensions)...")
train_textes_nomic = ["classification: " + text for text in X_train_text]
test_textes_nomic = ["classification: " + text for text in X_test_text]

t0 = time.time()
X_train_base = model.encode(train_textes_nomic, show_progress_bar=True)
X_test_base = model.encode(test_textes_nomic, show_progress_bar=True)
temps_encodage = time.time() - t0

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_base, y_train)
score_base = f1_score(y_test, clf.predict(X_test_base), average='weighted')

# =========================================================
# EXPERIMENTATION 2 : PCA (Réduction Linéaire à 64 dim)
# =========================================================
print("3. Application de la PCA (64 dimensions)...")
pca = PCA(n_components=64, random_state=42)
t0 = time.time()
X_train_pca = pca.fit_transform(X_train_base)
X_test_pca = pca.transform(X_test_base)
temps_pca = time.time() - t0

clf.fit(X_train_pca, y_train)
score_pca = f1_score(y_test, clf.predict(X_test_pca), average='weighted')

# =========================================================
# EXPERIMENTATION 3 : UMAP (Réduction Non-Linéaire à 64 dim)
# =========================================================
print("4. Application d'UMAP (64 dimensions)...")
reducer = umap.UMAP(n_components=64, random_state=42)
t0 = time.time()
X_train_umap = reducer.fit_transform(X_train_base)
X_test_umap = reducer.transform(X_test_base)
temps_umap = time.time() - t0

clf.fit(X_train_umap, y_train)
score_umap = f1_score(y_test, clf.predict(X_test_umap), average='weighted')

# =========================================================
# EXPERIMENTATION 4 : LE VRAI MATRYOSHKA (Truncation directe)
# =========================================================
print("5. Troncature Matryoshka (Coupe directe à 64 dimensions)...")
# C'est ici que la magie de Nomic opère : les premières dimensions concentrent l'info !
t0 = time.time()
X_train_matryoshka = X_train_base[:, :64]
X_test_matryoshka = X_test_base[:, :64]
temps_matryoshka = time.time() - t0 # Toujours instantané

clf.fit(X_train_matryoshka, y_train)
score_matryoshka = f1_score(y_test, clf.predict(X_test_matryoshka), average='weighted')

# =========================================================
# BILAN DES PERFORMANCES
# =========================================================
print("\n" + "="*65)
print(f"{'MÉTHODE':<25} | {'DIMENSIONS':<10} | {'SCORE F1':<10} | {'TEMPS RÉDUCTION'}")
print("="*65)
print(f"{'1. Nomic (Baseline)':<25} | {'768':<10} | {score_base:.4f}     | N/A (Encodage: {temps_encodage:.1f}s)")
print(f"{'2. PCA':<25} | {'64':<10} | {score_pca:.4f}     | {temps_pca:.4f}s")
print(f"{'3. UMAP':<25} | {'64':<10} | {score_umap:.4f}     | {temps_umap:.4f}s")
print(f"{'4. Nomic Matryoshka':<25} | {'64':<10} | {score_matryoshka:.4f}     | {temps_matryoshka:.4f}s (Immédiat)")
print("="*65)

c:\Users\titou\Documents\dev\Ynov\DataScience\Mini-memoire-Base-vectorielles\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Chargement des données (Espace vs Médecine)...

2. Chargement du modèle Nomic...


<All keys matched successfully>


Encodage des textes (Baseline à 768 dimensions)...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]